In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import datetime as datetime

file_path = 'data/F1_sample_data_with_random_consultoras.csv'  # Replace with your actual CSV file path
data = pd.read_csv(file_path)
print(data)

        CODEBELISTA  ANIOCAMPANA CODPAIS  CODPRODUCTOSAP      DESCATEGORIA  \
0          49958455       202302      PE       210100816        MAQUILLAJE   
1          50038750       202302      PE       200095170  CUIDADO PERSONAL   
2          26877393       202302      PE       200105125  CUIDADO PERSONAL   
3          48742556       202302      PE       200097066  CUIDADO PERSONAL   
4          49769164       202302      PE       200078924  CUIDADO PERSONAL   
...             ...          ...     ...             ...               ...   
168055     45658163       202418      PE       200105017        FRAGANCIAS   
168056     52166489       202418      PE       210103140            LENTES   
168057     52683025       202418      PE       200095931        FRAGANCIAS   
168058     52683025       202418      PE       200114054        MAQUILLAJE   
168059     52204089       202501      PE       210105381      COMPLEMENTOS   

       DESMARCA  PRECIOOFERTA FECHAPROCESO  
0         ESIKA   

In [2]:
# Convert FECHAPROCESO to datetime
data['FECHAPROCESO'] = pd.to_datetime(data['FECHAPROCESO'])

# Calculate Recency: Normalize such that 10 is the latest and 1 is the oldest for each consultora
latest_date = data['FECHAPROCESO'].max()
oldest_date = data['FECHAPROCESO'].min()

# For each consultora, calculate Recency and normalize it between 1 and 10
data['Recency'] = data.groupby('CODEBELISTA')['FECHAPROCESO'].transform(
    lambda x: 1 + 9 * ((latest_date - x).dt.days) / ((latest_date - oldest_date).days)
)

# Frequency by CODEBELISTA
frequency = data.groupby('CODEBELISTA').size().reset_index(name='Frequency')

# Normalize Frequency: Scale between 1 and 10
frequency['Frequency'] = 1 + 9 * (frequency['Frequency'] - frequency['Frequency'].min()) / (frequency['Frequency'].max() - frequency['Frequency'].min())

# Merge Frequency and Recency on CODEBELISTA
rf_data = pd.merge(frequency, data[['CODEBELISTA', 'Recency']].drop_duplicates(), on='CODEBELISTA')

# Final DataFrame
print(rf_data.head())

   CODEBELISTA  Frequency   Recency
0       227323   1.312655  7.464619
1       227323   1.312655  7.981308
2       227323   1.312655  9.567423
3      1058614   2.596774  7.440587
4      1058614   2.596774  7.152203


In [5]:
import numpy as np
import pandas as pd

df = pd.DataFrame(rf_data)

# Step 1: Normalize the data
normalized_data = (df - df.min()) / (df.max() - df.min())

# Step 2: Calculate proportions
proportions = normalized_data.div(normalized_data.sum(axis=0), axis=1)

# Step 3: Calculate entropy for each attribute
k = 1 / np.log(len(df))
entropy = -k * (proportions * np.log(proportions + 1e-12)).sum(axis=0)

# Step 4: Calculate degree of diversification
diversification = 1 - entropy

# Step 5: Compute weights
weights = diversification / diversification.sum()

# Extract alpha and beta
alpha, beta = weights['Frequency'], weights['Recency']
print(f"Alpha (Weight for Frequency): {alpha:.2f}")
print(f"Beta (Weight for Recency): {beta:.2f}")

# Combined score
df['Combined_Score'] = alpha * df['Frequency'] + beta * df['Recency']
print(df['Combined_Score'].head())


Alpha (Weight for Frequency): 0.68
Beta (Weight for Recency): 0.24
0    2.713803
1    2.839544
2    3.225539
3    3.585674
4    3.515494
Name: Combined_Score, dtype: float64
